# Sección 5 – MLP multi-salida, DLinear y NLinear en PyTorch

Este notebook entrena tres modelos de pronóstico:
- **MLPForecaster**: Perceptrón multicapa multi-salida
- **DLinear**: Descomposición lineal (tendencia + residuo)
- **NLinear**: Lineal con normalización de instancia

Se usa early stopping sobre el conjunto de validación y backtest rolling-origin.

In [ ]:
import sys
import os
import time
import math
import warnings
warnings.filterwarnings('ignore')

# Añadir raíz del proyecto al path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.dataset import ElectricityDataset
from src.models.mlp import MLPForecaster
from src.models.linear import DLinear, NLinear
from src.backtest import backtest
from src.metrics import evaluate_all, mae_per_horizon

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

## Hiperparámetros

In [ ]:
# ── Hiperparámetros globales ─────────────────────────────────────────────
L           = 168      # ventana de historia (horas)
H           = 24       # horizonte de pronóstico (horas)
BATCH_SIZE  = 512
HIDDEN_SIZE = 256
N_LAYERS    = 3
DROPOUT     = 0.1
LR          = 1e-3
WEIGHT_DECAY= 1e-4
EPOCHS      = 100
PATIENCE    = 10       # early stopping patience
STRIDE      = 24       # paso entre ventanas (un sample por día)

# Directorios de salida
CKPT_DIR    = os.path.join(ROOT, 'models', 'checkpoints')
OUT_DIR     = os.path.join(ROOT, 'outputs')
FIG_DIR     = os.path.join(OUT_DIR, 'figures')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)
os.makedirs(FIG_DIR,  exist_ok=True)
print('Directorios listos.')

## Carga del panel de electricidad

In [ ]:
# ── Cargar datos procesados ───────────────────────────────────────────────
DATA_PATH = os.path.join(ROOT, 'data', 'processed', 'electricity_panel.parquet')

if os.path.exists(DATA_PATH):
    panel = pd.read_parquet(DATA_PATH)
    print(f'Panel cargado: {panel.shape}  ({panel.index[0]} → {panel.index[-1]})')
else:
    # Fallback: generar datos sintéticos para demo
    print('[AVISO] Datos procesados no encontrados. Generando panel sintético (demo).')
    rng = np.random.default_rng(42)
    N_CLIENTS = 20
    idx = pd.date_range('2011-01-01', periods=35064, freq='h')
    vals = np.abs(
        rng.normal(loc=2.0, scale=0.8, size=(len(idx), N_CLIENTS)) +
        0.5 * np.sin(2 * np.pi * np.arange(len(idx))[:, None] / 24) +
        0.3 * np.sin(2 * np.pi * np.arange(len(idx))[:, None] / (24 * 7))
    ).clip(0)
    panel = pd.DataFrame(vals, index=idx, columns=[f'MT_{i:03d}' for i in range(N_CLIENTS)])
    print(f'Panel sintético: {panel.shape}')

# ── Splits temporales ─────────────────────────────────────────────────────
train_end = '2014-06-30 23:00'
val_end   = '2014-09-30 23:00'

panel_train = panel.loc[:train_end]
panel_val   = panel.loc[train_end:val_end].iloc[1:]  # evitar solapamiento
panel_test  = panel.loc[val_end:].iloc[1:]

print(f'Train: {panel_train.shape}  ({panel_train.index[0]} → {panel_train.index[-1]})')
print(f'Val  : {panel_val.shape}  ({panel_val.index[0]} → {panel_val.index[-1]})')
print(f'Test : {panel_test.shape}  ({panel_test.index[0]} → {panel_test.index[-1]})')

## Datasets y DataLoaders

In [ ]:
# ── Datasets ──────────────────────────────────────────────────────────────
ds_train = ElectricityDataset(panel_train, L=L, H=H, stride=STRIDE, use_covariates=True)
ds_val   = ElectricityDataset(panel_val, L=L, H=H, stride=STRIDE,
                               mean_=ds_train.mean_[0], std_=ds_train.std_[0],
                               use_covariates=True)

print(ds_train.window_count_info())
print(ds_val.window_count_info())
print(f'n_features: {ds_train.n_features}')

# ── DataLoaders ───────────────────────────────────────────────────────────
dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=0, pin_memory=(DEVICE.type == 'cuda'))
dl_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

print(f'Batches train: {len(dl_train)}  |  Batches val: {len(dl_val)}')

## Funciones genéricas de entrenamiento y validación

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Entrena el modelo por una época. Devuelve la pérdida media."""
    model.train()
    total_loss = 0.0
    n_batches  = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        preds = model(X)
        loss  = criterion(preds, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches  += 1
    return total_loss / n_batches


@torch.no_grad()
def validate(model, loader, criterion, device, dataset_val):
    """
    Valida el modelo. Devuelve (val_loss, MAE_val, MASE_val).
    MAE y MASE se calculan en escala NORMALIZADA (suficiente para early stopping).
    """
    model.eval()
    total_loss = 0.0
    n_batches  = 0
    all_preds, all_true = [], []
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        preds = model(X)
        total_loss += criterion(preds, y).item()
        n_batches  += 1
        all_preds.append(preds.cpu().numpy())
        all_true.append(y.cpu().numpy())
    all_preds = np.concatenate(all_preds, axis=0)
    all_true  = np.concatenate(all_true,  axis=0)
    mae_v  = float(np.mean(np.abs(all_preds - all_true)))
    # Naive seasonal MAE para MASE (escala normalizada)
    s = 24
    vals_norm = dataset_val.values_norm  # (T, N)
    naive_err = np.abs(vals_norm[s:] - vals_norm[:-s]).mean()
    mase_v = mae_v / (naive_err + 1e-8)
    return total_loss / n_batches, mae_v, float(mase_v)

## Loop de entrenamiento con early stopping

In [ ]:
def train_model(model, model_name, dl_train, dl_val, ds_val,
                n_epochs=EPOCHS, patience=PATIENCE,
                lr=LR, weight_decay=WEIGHT_DECAY, device=DEVICE):
    """
    Entrena cualquier modelo PyTorch con early stopping sobre val_loss.
    Devuelve historial de métricas.
    """
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=False
    )

    ckpt_path = os.path.join(CKPT_DIR, f'{model_name}_best.pt')
    history = []
    best_val_loss = float('inf')
    no_improve    = 0

    print(f'\n{'='*60}')
    print(f'Entrenando {model_name} | {sum(p.numel() for p in model.parameters()):,} params')
    print(f'{'='*60}')

    for epoch in range(1, n_epochs + 1):
        t0 = time.perf_counter()
        tr_loss           = train_epoch(model, dl_train, optimizer, criterion, device)
        val_loss, mae_v, mase_v = validate(model, dl_val, criterion, device, ds_val)
        elapsed           = time.perf_counter() - t0

        scheduler.step(val_loss)

        history.append({
            'epoch': epoch, 'train_loss': tr_loss, 'val_loss': val_loss,
            'MAE_val': mae_v, 'MASE_val': mase_v, 'tiempo_epoch': elapsed
        })

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1

        if epoch % 5 == 0 or no_improve == 0:
            print(f'  Epoch {epoch:3d}/{n_epochs} | '
                  f'train={tr_loss:.4f} val={val_loss:.4f} '
                  f'MAE={mae_v:.4f} MASE={mase_v:.4f} '
                  f't={elapsed:.1f}s  (no_improve={no_improve})')

        if no_improve >= patience:
            print(f'  Early stopping en epoch {epoch} (patience={patience})')
            break

    # Cargar mejores pesos
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f'  Mejor val_loss={best_val_loss:.4f} → pesos guardados en {ckpt_path}')
    return model, pd.DataFrame(history)

## Instanciar y entrenar los 3 modelos

In [ ]:
n_features = ds_train.n_features  # 8 (serie + 7 covariables de calendario)

# ── MLPForecaster ─────────────────────────────────────────────────────────
mlp_model = MLPForecaster(
    input_len=L, n_features=n_features,
    hidden_size=HIDDEN_SIZE, n_layers=N_LAYERS,
    output_len=H, dropout=DROPOUT
)
mlp_model, hist_mlp = train_model(mlp_model, 'mlp_best', dl_train, dl_val, ds_val)

# ── DLinear ───────────────────────────────────────────────────────────────
dlinear_model = DLinear(input_len=L, n_features=n_features, output_len=H, kernel_size=25)
dlinear_model, hist_dlinear = train_model(dlinear_model, 'dlinear_best', dl_train, dl_val, ds_val)

# ── NLinear ───────────────────────────────────────────────────────────────
nlinear_model = NLinear(input_len=L, output_len=H)
nlinear_model, hist_nlinear = train_model(nlinear_model, 'nlinear_best', dl_train, dl_val, ds_val)

## Curvas de pérdida

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
configs = [
    ('MLP',     hist_mlp,     '#2196F3'),
    ('DLinear', hist_dlinear, '#4CAF50'),
    ('NLinear', hist_nlinear, '#FF9800'),
]

for ax, (name, hist, color) in zip(axes, configs):
    ax.plot(hist['epoch'], hist['train_loss'], label='Train', color=color, lw=2)
    ax.plot(hist['epoch'], hist['val_loss'],   label='Val',   color=color, lw=2, ls='--')
    best_ep = hist.loc[hist['val_loss'].idxmin(), 'epoch']
    ax.axvline(best_ep, color='red', ls=':', lw=1.2, label=f'Best val (ep {best_ep})')
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Curvas de pérdida — MLP, DLinear, NLinear', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'loss_curves_mlp_linear.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada.')

## Wrapper PyTorchForecastModel para el protocolo de backtest

In [ ]:
class PyTorchForecastModel:
    """
    Adapta un modelo PyTorch al protocolo ForecastModel de src/backtest.py.
    
    El método predict() construye la ventana de entrada a partir del historial,
    ejecuta el modelo y desnormaliza los resultados.
    """

    def __init__(
        self,
        model: nn.Module,
        model_name: str,
        L: int,
        H: int,
        mean_: np.ndarray,   # (N,)
        std_: np.ndarray,    # (N,)
        device,
        use_covariates: bool = True,
    ):
        from src.dataset import build_calendar_features
        self.model          = model.eval().to(device)
        self._name          = model_name
        self.L              = L
        self.H              = H
        self.mean_          = mean_.reshape(1, -1).astype(np.float32)  # (1, N)
        self.std_           = (std_.reshape(1, -1) + 1e-8).astype(np.float32)
        self.device         = device
        self.use_covariates = use_covariates
        self._build_cal     = build_calendar_features

    @property
    def name(self) -> str:
        return self._name

    @torch.no_grad()
    def predict(
        self,
        history: pd.DataFrame,
        horizon: int,
        origin_time: pd.Timestamp,
    ) -> np.ndarray:
        """
        Returns:
            (N_clients, H) en kWh (escala original).
        """
        # ── Extraer ventana de historia (últimas L horas) ─────────────────
        hist_vals = history.values[-self.L:]          # (L, N)
        if hist_vals.shape[0] < self.L:
            pad = np.zeros((self.L - hist_vals.shape[0], hist_vals.shape[1]), dtype=np.float32)
            hist_vals = np.vstack([pad, hist_vals])

        # ── Normalización ─────────────────────────────────────────────────
        hist_norm = (hist_vals.astype(np.float32) - self.mean_) / self.std_  # (L, N)
        N = hist_norm.shape[1]

        if self.use_covariates:
            cal_idx = history.index[-self.L:]
            if len(cal_idx) < self.L:
                fill_idx = pd.date_range(
                    end=cal_idx[0] - pd.Timedelta(hours=1),
                    periods=self.L - len(cal_idx), freq='h'
                )
                cal_idx = fill_idx.append(cal_idx)
            cal = self._build_cal(cal_idx)  # (L, 7)
            # Construir X para cada cliente: (N, L, 8)
            X_all = np.concatenate(
                [hist_norm.T[:, :, None], np.tile(cal[None], (N, 1, 1))], axis=-1
            ).astype(np.float32)
        else:
            X_all = hist_norm.T[:, :, None].astype(np.float32)  # (N, L, 1)

        X_tensor = torch.from_numpy(X_all).to(self.device)  # (N, L, F)

        # ── Inferencia ────────────────────────────────────────────────────
        preds_norm = self.model(X_tensor).cpu().numpy()  # (N, H)

        # ── Desnormalización ──────────────────────────────────────────────
        preds = preds_norm * self.std_.T + self.mean_.T   # (N, H)
        preds = np.clip(preds, 0, None)
        return preds  # (N, H)

## Backtest rolling-origin (oct–dic 2014)

In [ ]:
# Serie de entrenamiento en escala original para MASE
y_train_orig = panel_train.values.T.astype(np.float32)  # (N, T_train)

results = {}
all_predictions = []

for model_obj, model_name in [
    (mlp_model,     'MLPForecaster'),
    (dlinear_model, 'DLinear'),
    (nlinear_model, 'NLinear'),
]:
    print(f'\nBacktest → {model_name}')
    wrapper = PyTorchForecastModel(
        model=model_obj,
        model_name=model_name,
        L=L, H=H,
        mean_=ds_train.mean_[0],
        std_=ds_train.std_[0],
        device=DEVICE,
        use_covariates=True,
    )
    res = backtest(
        model=wrapper,
        panel=panel,
        y_train=y_train_orig,
        H=H,
        test_start='2014-10-01',
        test_end='2014-12-31',
        seasonality=24,
        return_predictions=True,
        verbose=True,
    )
    results[model_name] = res
    if res['predictions'] is not None:
        all_predictions.append(res['predictions'])

print('\nBacktest completo.')

## Tabla de métricas

In [ ]:
metrics_rows = []
for name, res in results.items():
    row = dict(res['metrics'])
    row['inference_time_s'] = res.get('inference_time', 0)
    metrics_rows.append(row)

df_metrics = pd.DataFrame(metrics_rows).set_index('model')
df_metrics.index.name = 'Modelo'
print(df_metrics.to_string(float_format='{:.4f}'.format))

# Guardar CSV
metrics_path = os.path.join(OUT_DIR, 'metrics_mlp_linear.csv')
df_metrics.to_csv(metrics_path)
print(f'\nMétricas guardadas en: {metrics_path}')

## Guardar predicciones

In [ ]:
if all_predictions:
    df_preds = pd.concat(all_predictions, ignore_index=True)
    preds_path = os.path.join(OUT_DIR, 'predictions_mlp_linear.csv')
    df_preds.to_csv(preds_path, index=False)
    print(f'Predicciones guardadas: {df_preds.shape}  →  {preds_path}')
    display(df_preds.head())
else:
    print('No hay predicciones para guardar.')

## MAE por paso del horizonte

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
colors = {'MLPForecaster': '#2196F3', 'DLinear': '#4CAF50', 'NLinear': '#FF9800'}

for model_name, res in results.items():
    mae_h = res['mae_horizon']
    ax.plot(range(1, H+1), mae_h, marker='o', ms=4,
            label=model_name, color=colors.get(model_name), lw=2)

ax.set_xlabel('Paso del horizonte (h)', fontsize=12)
ax.set_ylabel('MAE (kWh)', fontsize=12)
ax.set_title('MAE por paso del horizonte — MLP, DLinear, NLinear', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'mae_horizon_mlp_linear.png'), dpi=150, bbox_inches='tight')
plt.show()

## Resumen del historial de entrenamiento

In [ ]:
for name, hist in [('MLP', hist_mlp), ('DLinear', hist_dlinear), ('NLinear', hist_nlinear)]:
    best = hist.loc[hist['val_loss'].idxmin()]
    print(f'{name}:')
    print(f'  Épocas entrenadas : {len(hist)}')
    print(f'  Mejor epoch       : {int(best.epoch)}')
    print(f'  Mejor val_loss    : {best.val_loss:.5f}')
    print(f'  MAE_val (norm)    : {best.MAE_val:.5f}')
    print(f'  MASE_val (norm)   : {best.MASE_val:.5f}')
    print(f'  Tiempo/época (s)  : {hist.tiempo_epoch.mean():.2f}')
    print()